# Limpeza de Áudio por Predição do LSTM

Objetivo: remover trechos preditos como ruido/silencio e exportar WAV limpo para o notebook de transcrição.

Observação importante: este notebook assume que o CSV de features e o WAV referem-se ao mesmo sinal temporal.

In [28]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import soundfile as sf

from chime_lstm_data import build_sequences
from lstm_model import LSTMClassifier, infer_records, load_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [34]:
BASE_DIR = Path('.')
ARTIFACTS_DIR = BASE_DIR / 'results' / 'lstm_vad'
OUTPUT_DIR = BASE_DIR / 'results' / 'clean_audio'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ajuste para o arquivo que deseja limpar. Cada .wav tem um .csv correspondente
wav_filename = 'S01_U06.CH1.wav'
csv_filename = 'chime6_kinect_eval.csv'
CSV_PATH = BASE_DIR / csv_filename
WAV_PATH = BASE_DIR / 'audio' / 'eval' / wav_filename
MODEL_PATH = ARTIFACTS_DIR / 'lstm_vad_best.pt'

DATA_CONFIG_PATH = ARTIFACTS_DIR / 'metrics_summary.json'
BEST_PARAMS_PATH = ARTIFACTS_DIR / 'best_params.json'
NORMALIZATION_CONFIG_PATH = ARTIFACTS_DIR / 'normalization_config.json'

DEFAULT_THRESHOLD = 0.5
FADE_MS = 16

print('WAV alvo:', WAV_PATH)

WAV alvo: audio\eval\S01_U06.CH1.wav


## 1. Carregar artefatos do treino

In [39]:
with open(NORMALIZATION_CONFIG_PATH, encoding='utf-8') as f:
    scaler_cfg = json.load(f)
    
with open(DATA_CONFIG_PATH, encoding='utf-8') as f:
    data_cfg = json.load(f)
    threshold = data_cfg.get('score_threshold', DEFAULT_THRESHOLD)

with open(BEST_PARAMS_PATH, encoding='utf-8') as f:
    best_params = json.load(f)

feature_cols = scaler_cfg['feature_cols']
seq_len = int(data_cfg['seq_len'])
stride = int(best_params['stride'])

model = LSTMClassifier(
    input_dim=len(feature_cols),
    hidden_dim=int(best_params['hidden_dim']),
    num_layers=int(best_params['num_layers']),
    dropout_rate=float(best_params['dropout_rate']),
    output_dim=1
)
model = load_model(model, str(MODEL_PATH), device)
print('Modelo carregado.')

Modelo carregado.


## 2. Carregar features do sample e inferir fala/ruído

In [40]:
df = pd.read_csv(CSV_PATH)

wav_stem = Path(wav_filename).stem
sample_prefix = wav_stem.split('.CH')[0]
if 'sample_id' not in df.columns:
    raise ValueError(f"CSV sem coluna 'sample_id': {CSV_PATH}")

sample_ids = df['sample_id'].astype(str)
sample_mask = sample_ids.str.startswith(sample_prefix)
if not sample_mask.any():
    available_samples = sample_ids.drop_duplicates().head(10).tolist()
    raise ValueError(
        f"Nenhuma linha encontrada para o prefixo '{sample_prefix}' em {CSV_PATH}. "
        f"Exemplos disponíveis: {available_samples}"
    )

df = df.loc[sample_mask].sort_values(['sample_id', 'timestamp_ms']).reset_index(drop=True)

if df.empty:
    raise ValueError(f'Sample não encontrado: {CSV_PATH}')

pack = build_sequences(
    df=df,
    feature_cols=feature_cols,
    seq_len=seq_len,
    stride=stride,
    label_mode='last',
)

preds, scores = infer_records(model, pack.X, threshold=threshold, device=device)
print('prefixo alvo:', sample_prefix)
print('sample_ids usados:', df['sample_id'].nunique())
print('frames do sample:', len(df))
print('records:', len(preds), 'speech_ratio:', float(preds.mean()) if len(preds) else 0.0)

prefixo alvo: S01_U06
sample_ids usados: 318
frames do sample: 1192818
records: 11766 speech_ratio: 0.6980282168961415


## 3. Converter predição por record em máscara por frame

In [41]:
n_frames = len(df)
score_sum = np.zeros(n_frames, dtype=np.float32)
score_cnt = np.zeros(n_frames, dtype=np.float32)

seq_cursor = 0
for _, g in df.groupby('sample_id', sort=False):
    group_index = g.index.to_numpy()
    group_len = len(group_index)
    if group_len < seq_len:
        continue

    for start in range(0, group_len - seq_len + 1, stride):
        if seq_cursor >= len(scores):
            break
        end = start + seq_len
        window_index = group_index[start:end]
        score_sum[window_index] += float(scores[seq_cursor])
        score_cnt[window_index] += 1.0
        seq_cursor += 1

frame_score = np.zeros(n_frames, dtype=np.float32)
valid = score_cnt > 0
frame_score[valid] = score_sum[valid] / score_cnt[valid]

# Usa o threshold configurado no score médio por frame.
frame_mask = np.zeros(n_frames, dtype=np.float32)
frame_mask[valid] = (frame_score[valid] >= float(threshold)).astype(np.float32)

# Frames sem voto herdam o último estado conhecido dentro de cada bloco.
for _, g in df.groupby('sample_id', sort=False):
    group_index = g.index.to_numpy()
    group_valid = valid[group_index]
    if not np.any(group_valid):
        continue
    last_pos = np.where(group_valid)[0][-1]
    last_score = frame_score[group_index[last_pos]]
    trailing_index = group_index[~group_valid]
    if len(trailing_index):
        frame_score[trailing_index] = last_score
        frame_mask[trailing_index] = float(last_score >= float(threshold))

print('seqs mapeadas:', int(seq_cursor), 'de', int(len(scores)))
print('frame speech ratio:', float(frame_mask.mean()))
print('frame score range:', round(float(frame_score.min()), 4), '->', round(float(frame_score.max()), 4))
print('threshold aplicado:', float(threshold))

seqs mapeadas: 11766 de 11766
frame speech ratio: 0.6980293989181519
frame score range: 0.0128 -> 0.8497
threshold aplicado: 0.72


## 4. Aplicar máscara no áudio e exportar

In [43]:
audio, sr = sf.read(WAV_PATH)
if audio.ndim > 1:
    audio = audio.mean(axis=1)

timestamps_ms = df['timestamp_ms'].to_numpy(dtype=np.float64)

# Estima passo temporal por frame via mediana.
if len(timestamps_ms) > 1:
    frame_step_ms = float(np.median(np.diff(timestamps_ms)))
else:
    frame_step_ms = 8.0

samples_per_frame = max(1, int(round((frame_step_ms / 1000.0) * sr)))
keep_mask = np.zeros(len(audio), dtype=bool)

for i, keep in enumerate(frame_mask.astype(bool)):
    a = i * samples_per_frame
    b = min(len(audio), (i + 1) * samples_per_frame)
    if a >= len(audio):
        break
    keep_mask[a:b] = keep

# Garante que o rabicho final do áudio herde o último estado conhecido.
last_covered_sample = min(len(audio), n_frames * samples_per_frame)
if last_covered_sample < len(audio) and len(frame_mask) > 0:
    keep_mask[last_covered_sample:] = bool(frame_mask[-1])

keep_ratio = float(keep_mask.mean()) if len(keep_mask) else 0.0

# Para threshold <= 0, o comportamento esperado é preservar o áudio original.
if float(threshold) <= 0.0:
    clean = audio.copy()
else:
    # Recorta os trechos descartados em vez de apenas silenciar, reduzindo o tamanho do WAV.
    clean = audio[keep_mask]

out_suffix = '_trimmed.wav' if float(threshold) > 0.0 else '_original.wav'
out_wav = OUTPUT_DIR / wav_filename.replace('.wav', out_suffix)
sf.write(out_wav, clean, sr)

original_duration_s = len(audio) / sr if sr else 0.0
clean_duration_s = len(clean) / sr if sr else 0.0
print('Áudio salvo em:', out_wav)
print('keep_ratio:', round(keep_ratio, 4))
print('duration original (s):', round(original_duration_s, 3))
print('duration clean (s):', round(clean_duration_s, 3))

Áudio salvo em: results\clean_audio\S01_U06.CH1_trimmed.wav
keep_ratio: 0.6982
duration original (s): 9546.976
duration clean (s): 6665.408
